In [ ]:
# for matplotlib, Set ggplot theme
import matplotlib.pyplot as plt
plt.style.use('ggplot')


# FORELÆSNING 12: Supervised learning – klassifikation

> k-Nearest Neighbors, feature-skalering, beslutningsgrænser, modelevaluering og krydsvalidering.

### Underviser: Martin Siemienski Andersen mvan@hst.aau.dk

**ST2 - Anvendt Programmering**

In [ ]:
from IPython.display import HTML
import requests

url = "https://raw.githubusercontent.com/AAU-ST2-Programming/all_lectures/refs/heads/main/overview_files/shared_overview_table12.html"
html = requests.get(url, timeout=30).text
HTML(html)

# Unsupervised learning vs. supervised learning
- **Unsupervised learning**: Læring uden mærkede data. Modellen forsøger at finde mønstre eller strukturer i dataene uden nogen foruddefinerede etiketter.
- **Supervised learning**: Læring med mærkede data. Modellen trænes på et datasæt, hvor hver indgang har en tilknyttet etiket, og målet er at lære en funktion, der kan forudsige etiketterne for nye, usete data.

# Clustering vs. Klassifikation vs. Regression
**Regression**: En type supervised learning
- Målet er at forudsige en kontinuerlig værdi for hver indgang. 
- Eksempler inkluderer at forudsige huspriser baseret på forskellige funktioner eller at forudsige temperaturen baseret på historiske data.

**Klassifikation**: En type supervised learning  
- Målet er at forudsige en diskret etiket eller klasse for hver indgang. 
- Eksempler inkluderer at klassificere e-mails som spam eller ikke-spam, eller at genkende håndskrevne cifre.

**Clustering**: En type unsupervised learning
- Målet er at gruppere data i klynger baseret på ligheder uden at have foruddefinerede labels. 
- Eksempler inkluderer at segmentere kunder i forskellige grupper baseret på deres købsadfærd eller at identificere mønstre i geografiske data.




# k-Nearest Neighbors (k-NN): Introduktion


- Simpel algoritme til klassifikation
- Finder de 'k' nærmeste naboer
- Stemmer om klassen
- Ingen træning – bruger data direkte


## Sådan gør du (trin-for-trin)

1. Vælg antal naboer (k)
2. Mål afstand til alle punkter
3. Find de k nærmeste
4. Stem om klassen
5. (Valgfrit) Skaler features


## Visuelt eksempel på k-NN i 2D

Her viser vi, hvordan k-NN virker på et meget simpelt 2D-eksempel, hvor vi kan tegne punkterne og beslutningsgrænsen.

## Simpelt 2D k-NN eksempel (kode og plot)

Vi genererer nogle simple data, træner k-NN og viser beslutningsgrænsen og fordelingen af klasser.

# Simpelt 2D eksempel med k-NN (manuel kode)
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Brug scikit-learn til at lave simple 2D data
X_all, y_all = make_classification(n_samples=30,n_features=3)

# Funktion til at forudsige klassen for et punkt med k-NN (manuel)
def manual_knn_predict(X_train, y_train, new_point, k=3):
    distances = np.sqrt(np.sum((X_train - new_point)**2, axis=1))
    nearest_idx = np.argsort(distances)[:k]
    votes = y_train[nearest_idx]
    predicted_class = np.argmax(np.bincount(votes))
    return predicted_class, nearest_idx

# Tegn beslutningsgrænse
h = .05
x_min, x_max = X_all[:, 0].min() - 1, X_all[:, 0].max() + 1
y_min, y_max = X_all[:, 1].min() - 1, X_all[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = np.zeros(xx.shape)
for i in range(xx.shape[0]):
    for j in range(xx.shape[1]):
        Z[i, j], _ = manual_knn_predict(X_all, y_all, np.array([xx[i, j], yy[i, j]]), k=3)

plt.figure(figsize=(7,5))
plt.contourf(xx, yy, Z, cmap=plt.cm.Pastel1, alpha=0.7)
plt.scatter(X_all[:, 0], X_all[:, 1], c=y_all, cmap=plt.cm.Set1, edgecolor='k', s=80)

# Eksempel: klassificér et nyt punkt
new_point = np.array([1, -1])
pred_class, nearest_idx = manual_knn_predict(X_all, y_all, new_point, k=3)
plt.scatter(new_point[0], new_point[1], c='gold', edgecolor='black', s=120, marker='*', label='Nyt punkt')
plt.scatter(X_all[nearest_idx,0], X_all[nearest_idx,1], facecolors='none', edgecolors='lime', s=200, linewidths=2, label='Nærmeste naboer')
plt.title(f'Manuel k-NN: Klassificeret som {pred_class}')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.show()

## Interaktivt k-NN eksempel: Flyt et punkt og se klassifikation

Her kan du flytte et nyt punkt rundt i 2D og se, hvilke k naboer der vælges, og hvilken klasse punktet får.

In [ ]:
# Interaktivt k-NN plot med sliders
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider
from sklearn.datasets import make_blobs
import numpy as np

# Generér simple 2D data med labels
X_simple, y_simple = make_blobs(n_samples=30, centers=2, n_features=2, cluster_std=2.2, random_state=42)


# Sliders
s_x = FloatSlider(min=X_simple[:,0].min()-1, max=X_simple[:,0].max()+1, value=0)
s_y = FloatSlider(min=X_simple[:,1].min()-1, max=X_simple[:,1].max()+1, value=0)
s_k = IntSlider(min=1, max=10, value=3)


# Manuel k-NN funktion
def manual_knn_predict(X_train, y_train, new_point, k=3):
    distances = np.sqrt(np.sum((X_train - new_point)**2, axis=1))
    nearest_idx = np.argsort(distances)[:k]
    votes = y_train[nearest_idx]
    predicted_class = np.argmax(np.bincount(votes))
    return predicted_class, nearest_idx

# Funktion til at opdatere plot
highlight = None
C  = {0: 'red', 1: 'blue', 2: 'green', 3: 'magenta'}# farver efter std color sceme
def update(x,y,k):
    plt.figure(figsize=(7,5))
    pred_class, idxs = manual_knn_predict(X_simple, y_simple, np.array([x, y]), k)
    # Fremhæv de k nærmeste naboer
    highlight = [plt.scatter(X_simple[idxs,0], X_simple[idxs,1], facecolors='none', edgecolors='lime', s=200, linewidths=2, label='Nærmeste naboer')]
    plt.scatter(X_simple[:,0], X_simple[:,1], c=[C[label] for label in y_simple], edgecolor='k', s=80, label='Data punkter')
    
    plt.scatter(x, y, c=C[pred_class], edgecolor='black', s=120, marker='*', label='Nyt punkt')

    plt.title(f"K-NN iteration {k}")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()

    plt.show()


interact(update, x=s_x,y=s_y, k=s_k)

# Indlæs og udforsk data

Vi bruger brystkræftdatasættet fra scikit-learn til at demonstrere k-NN klassifikation, beslutningsgrænser og modelevaluering.

In [ ]:
# Import libraries

from sklearn.datasets import load_breast_cancer

# Load dataset
data = load_breast_cancer()
X = data.data
y = data.target
print("Data Keys:", list(data.keys()))
print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Target names: {data.target_names}")
print(f"Feature names: {data.feature_names[:5]} ...")

In [ ]:

import matplotlib.pyplot as plt
# Select two features for visualization
feature_x = 0  # mean radius
feature_y = 1  # mean texture

plt.figure(figsize=(8,6))
for label, color, name in zip([0,1], ['red', 'blue'], data.target_names):
    plt.scatter(X[y==label, feature_x], X[y==label, feature_y],
                label=name, alpha=0.5, c=color)
plt.xlabel(data.feature_names[feature_x])
plt.ylabel(data.feature_names[feature_y])
plt.title('Breast Cancer Data: Two Features')
plt.legend()
plt.show()

# Hvorfor splitte data i træning og test?


- Testdata simulerer nye, usete data.
- Vi undgår at modellen bare "husker" træningsdata (overfitting).
- Giver et ærligt billede af modellens præstation i praksis.
- Tester om modellen kan generalisere til nye data.

In [ ]:

from sklearn.model_selection import train_test_split
# Step 1: Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training samples: {X_train.shape}")
print(f"Test samples: {X_test.shape}")

# Scikit-learns k-NN
>```python
>from sklearn.neighbors import KNeighborsClassifier
>model = KNeighborsClassifier(n_neighbors=3)
>model.fit(X, y)
>```

# Øvelse: Kaffe, Kage og Klynger – KNN på brystkræftdata ☕🍰


**Filnavn-forslag:** `ex_kaffeklynge.py`

### Scenarie
Forestil dig, at du arbejder i en lettere kaotisk (men sjov) virksomhed. Du får ofte mærkelige opgaver – men i dag er det faktisk spændende! Du er blevet bedt om at undersøge, hvordan man kan bruge maskinlæring til at klassificere brystkræft baseret på data fra scikit-learn.


### Hvad du ved:
- Hvad supervised learning er, og hvorfor vi splitter data i træning/test
- Hvordan k-NN virker, og hvordan man bruger det til klassifikation
- At det er vigtigt at teste på usete data for at vurdere generalisering


### Opgave
- Brug kun de første to features fra `breast_cancer` datasættet fra scikit-learn.
- Del data i trænings- og testdata.
- Træn en k-NN model og predikter på testdata.
    - Hvor mange % af testdata klassificeres korrekt?
    - Hvad er beslutningsgrænsen for jeres model?
- Overvej: Hvilke features ville du ellers vælge, hvis du måtte vælge frit?


*Hold det simpelt – prøv dig frem og hav det sjovt! Vi kommer til at arbejde med analyser af modelperformance om lidt, så ikke brug for meget tid på det.*

## Svar:

# KNN på brystkræftdata: Træning, forudsigelse og evaluering
- Indlæs data
- Split i træning og test
- Træn k-NN
- Forudsig på testdata
- Evaluer præstation


## Evaluering af modelpræstation


- **Nøjagtighed (accuracy):** Andel af korrekte forudsigelser
- **Præcision (precision):** Hvor mange af de positive forudsigelser var rigtige?
- **Sensitivitet (recall/sensitivity):** Hvor mange af de sande positive blev fundet? (også kaldet "recall")
- **Specificitet (specificity):** Hvor mange af de sande negative blev fundet?

Disse mål beregnes ud fra en *confusion matrix* (forvirringsmatrix).

Vi kan bruge `sklearn.metrics` til at beregne dem.

# Confusion matrix – hvad er det?


- En **confusion matrix** (forvirringsmatrix) viser, hvordan en klassifikationsmodel klarer sig på testdata.
- Den sammenligner de sande klasser (faktiske) med modellens forudsigelser.


|                | Forudsagt positiv | Forudsagt negativ |
|----------------|------------------|------------------|
| **Faktisk positiv** | True Positive (TP)   | False Negative (FN)  |
| **Faktisk negativ** | False Positive (FP)  | True Negative (TN)   |


- Bruges til at beregne mål som sensitivitet og specificitet.

## I python
>```python
>from sklearn.metrics import confusion_matrix
>cm = confusion_matrix(y_test, y_pred)
>tn, fp, fn, tp = cm.ravel()
>```

# Sensitivitet (Recall)


- **Sensitivitet** (også kaldet *recall*) måler hvor stor en andel af de sande positive, som modellen korrekt finder.
- Formel:  
  $\text{Sensitivitet} = \frac{\text{True Positives}}{\text{True Positives} + \text{False Negatives}}$


- I confusion matrixen: Hvor mange af de faktiske positive (syge) bliver korrekt klassificeret som positive?

## I Python
>```python
>from sklearn.metrics import recall_score
>recall = recall_score(y_test, y_pred)
>```

# Specificitet


- **Specificitet** måler hvor stor en andel af de sande negative, som modellen korrekt finder.
- Formel:  
  $\text{Specificitet} = \frac{\text{True Negatives}}{\text{True Negatives} + \text{False Positives}}$


- I confusion matrixen: Hvor mange af de faktiske negative (raske) bliver korrekt klassificeret som negative?

## I Python
>```python
>specificity = tn / (tn + fp)
>```

# Python-kode til confusion matrix, sensitivitet og specificitet, samlet

In [ ]:
# Beregn og vis forskellige præstationsmål
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from sklearn.neighbors import KNeighborsClassifier


def print_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    sensitivity = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp)
    print(f"Nøjagtighed (accuracy): {accuracy:.2f}")
    print(f"Præcision (precision): {precision:.2f}")
    print(f"Sensitivitet (recall/sensitivity): {sensitivity:.2f}")
    print(f"Specificitet (specificity): {specificity:.2f}")
    print(f"Confusion matrix:\n{cm}")


model = KNeighborsClassifier(n_neighbors=1)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print_metrics(y_test, y_pred)

# Visualizering af confusion matrix

In [ ]:

from sklearn.metrics import ConfusionMatrixDisplay


ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title('Confusion Matrix')
plt.grid(False)  # Fjern gitterlinjer for bedre læsbarhed
plt.show()